In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, random_split, DataLoader, Subset

import string
import unicodedata

from io import open
import glob
import os

import random
import numpy as np

from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt

In [17]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

torch.set_default_device(device)

# Loading & Preprocessing

### Cleaning characters (ASCII only)

In [18]:
allowed_characters = string.ascii_letters + "_,.;" # vocabulary of allowed characters 
n_characters = len(allowed_characters)

def convertToASCII(string):
    # filtering only valid values like alphabets and specificed symobls
    return "".join(
        character for character in unicodedata.normalize("NFC", string)
        if unicodedata.category(character) != "Mn" and character in allowed_characters
    )

### Conversion to tensors

In [19]:
def characterToIndex(char):
    if char not in allowed_characters:
        return allowed_characters.find("_")
    else:
        # returning index of character in vocabulary a => 0, b => 1, ...
        return allowed_characters.find(char)

# Line has one name
# Each name has to be reprensented as 3d matrix with each letter one hot encoded
# 1d represent word, 2d represent batch size; 3d represent character
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_characters)
    for index, letter in enumerate(line):
        # for each letter turning on it's letter column
        tensor[index][0][characterToIndex(letter)] = 1
    return tensor


### Convert to dataset

In [20]:
class NamesDataset(Dataset):

    def __init__(self, data_dir):
        super().__init__()

        self.data_dir = data_dir
        labels_set = set()

        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        text_files = glob.glob(os.path.join(data_dir, "*.txt")) # reading all text files in the specificed directory
        for filename in text_files:
            # label is given by name of file
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            # each file has many lines, each line has one name
            lines = open(filename, encoding="utf-8").read().strip().split("\n") 
            for name in lines:
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label)

            self.labels_uniq = list(labels_set)
            for idx in range(len(self.labels)):
                temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
                self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data = self.data[idx]
        label = self.labels[idx]
        data_tensor = self.data_tensors[idx]
        label_tensor = self.labels_tensors[idx]

        return label_tensor, data_tensor, label, data




### Loading the dataset

In [45]:
dataset = NamesDataset("datasets/names")
shuffled_indices = torch.randperm(len(dataset)).tolist()
shuffled_dataset = Subset(dataset, shuffled_indices)
train_set, val_set, test_set = random_split(shuffled_dataset, [0.7, 0.15, 0.15,])

# CNN Network

### Define network

In [22]:
class NamesRNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Sequential(
            nn.Linear(
                hidden_size,
                output_size,
            ),
            nn.LogSoftmax(
                dim=1
            )
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        output, _ = self.rnn(x, h0)
        return self.fc(output[-1])
        
    

### Build the network

In [23]:
rnn_model = NamesRNN(
    n_characters,
    128,
    len(dataset.labels_uniq)
)

In [24]:
def output_to_label(output, output_labels):
    _, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i]

### Train the network

In [25]:
batch_size = 64
criterion = nn.NLLLoss()
file_path="rnn_name_classifier"

In [42]:
def train(rnn_model, train_set, val_set, n_epochs=10, learning_rate=0.001, batch_size=batch_size, criterion=criterion, file_path=file_path):
    min_val_loss = float("inf")
    optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

    for epoch in range(0, n_epochs):
        train_batches = list(range(len(train_set)))
        random.shuffle(train_batches)
        train_batches = [
            train_batches[i: batch_size + i]
            for i in range(0, len(train_batches), batch_size)
        ]
    
        val_batches = list(range(len(val_set)))
        random.shuffle(val_batches)
        val_batches = [
            val_batches[i: batch_size + i]
            for i in range(0, len(val_batches), batch_size)
        ]

        # Train
        rnn_model.train()
        train_loss = 0

        for index, batch in enumerate(train_batches):
            batch_loss = 0

            for i in batch:
                (label_tensor, text_tensor, label, text) = train_set[i]
                label_tensor_pred = rnn_model(text_tensor)
                loss = criterion(label_tensor_pred, label_tensor)
                batch_loss += loss

            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn_model.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()

            train_loss += batch_loss.item() / (len(batch))

        # Validate
        rnn_model.eval()
        val_loss = 0

        with torch.no_grad():
            for index, batch in enumerate(val_batches):
                batch_loss = 0
            
                for i in batch:
                    (label_tensor, text_tensor, label, text) = val_set[i]
                    label_tensor_pred = rnn_model(text_tensor)
                    loss = criterion(label_tensor_pred, label_tensor)
                    batch_loss += loss
                    
                val_loss += batch_loss.item()

        train_loss /= len(train_batches)
        val_loss /= len(val_batches)
        
        if val_loss < min_val_loss:
            min_val_loss = val_loss
            torch.save(rnn_model.state_dict(), f"{file_path}.pt")

        print(f"Epoch: {epoch} Train loss: {train_loss} Val loss: {val_loss}")

train(rnn_model, train_set, val_set)


Epoch: 0 Train loss: 1.3802482737435235 Val loss: 86.87803401549657
Epoch: 1 Train loss: 1.3798596484492522 Val loss: 87.12116585175197
Epoch: 2 Train loss: 1.3792801520439109 Val loss: 86.91306874155998
Epoch: 3 Train loss: 1.3789168670924021 Val loss: 86.82669201493263
Epoch: 4 Train loss: 1.3796818275042255 Val loss: 86.66816481947899
Epoch: 5 Train loss: 1.3791825517861531 Val loss: 86.69756622612476
Epoch: 6 Train loss: 1.379764683680101 Val loss: 86.92955972750981
Epoch: 7 Train loss: 1.3801074904022794 Val loss: 87.1661028067271
Epoch: 8 Train loss: 1.3786749290697502 Val loss: 87.44233040014903
Epoch: 9 Train loss: 1.3795935668728567 Val loss: 86.80365683635075


### Evaluate the model

In [43]:
rnn_model.load_state_dict(
    torch.load("rnn_name_classifier.pt")
)

<All keys matched successfully>

In [46]:
rnn_model.eval()

test_batches = list(range(len(val_set)))
random.shuffle(test_batches)
test_batches = [
    test_batches[i: batch_size + i]
    for i in range(0, len(test_batches), batch_size)
]

y_actual = []
y_pred = []

with torch.no_grad():

    for index, batch in enumerate(test_batches):
        batch_loss = 0
        
        for i in batch:
            (label_tensor, text_tensor, label, text) = test_set[i]
            label_tensor_pred = rnn_model(text_tensor)
            y_actual.append(output_to_label(label_tensor, dataset.labels_uniq))
            y_pred.append(output_to_label(label_tensor_pred, dataset.labels_uniq))

In [47]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.020923281301893058
              precision    recall  f1-score   support

     Chinese       0.00      0.00      0.00         0
      German       0.00      0.00      0.00         0
     Italian       1.00      0.02      0.04      3011
  Portuguese       0.00      0.00      0.00         0

    accuracy                           0.02      3011
   macro avg       0.25      0.01      0.01      3011
weighted avg       1.00      0.02      0.04      3011



c:\Users\User\.conda\envs\torch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\.conda\envs\torch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\.conda\envs\torch_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
